# MS3: EDA, Baseline Modeling, and Pipeline Development

**Canvas Project #150**

**Project:** Hyperbolic Embeddings for Hierarchical Style Representation

**Group Members:** Peter Flo, Luca Grossmann, Valerie Wang

**COMPSCI 2090B — Spring 2026**

## 1. Problem Statement Refinement and Introduction

Develop an introduction to your project (can be a brief summary of your Milestone 2)

Reiterate/refine your problem statement if needed. Ensure it is clear and understandable to someone outside your field.

Articulate significance based on new insights from EDA, highlighting shifts in perspective or understanding.

## 2. Comprehensive EDA Review:
Include key findings from your extended EDA, focusing on how these insights informed feature engineering and model selection.

Detail the feature engineering process, including transformations, encodings, or selection techniques, with justifications rooted in EDA.

Include visualizations that support analytical decisions, linking back to the problem statement and project goals.

## 3. Baseline Model Selection and Justification:
Explain the choice of your baseline model, considering simplicity, interpretability, and relevance with respect to your project statement.

Define the data used for training and possibly testing. Be sure to address whether entire data set was considered, and if not, why.

Detail the training process, including preprocessing, parameter choices and/or tuning, and metrics used for evaluation.

### Baseline Model

### Data Preprocessing

To preporcess the data for our baseline model, we had to prepare the data for the CLIP ViT-B/16 image encoder which we are going to keep frozen during training.

To do this, we:

1. **Resize** the shorter side to 224.
2. **CenterCrop** to 224×224.
3. **Convert to RGB** 
4. **ToTensor** — `PIL` → `float32` tensor `(3, 224, 224)`.
5. **Normalize** with CLIP's mean/std:
   - mean = `(0.48145466, 0.4578275, 0.40821073)`
   - std  = `(0.26862954, 0.26130258, 0.27577711)`

In addition to this, we set up a 70/30 train/test/val split. We have additional data processing work that we will need to include in our workflow (class balancing, class masking, data augmentation). However, for the baseline we decided it would be best to start with a fairly raw dataset. 



### Embedding

To avoid passing the data through Clip on every forward pass, we did one pass through of our entire dataset and saved the output. 

The outputted data is much lower dimensional, and has (some) emeded semantic feauteres of the dataset. 

``` Python
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-16", pretrained="openai", device=device)
    model.eval()

    loader = DataLoader(WikiArtDataset(paths, preprocess),
                        batch_size=128, num_workers=8, shuffle=False)

    chunks = []
    t0 = time.time()
    with torch.no_grad():
        for batch in tqdm(loader):
            chunks.append(model.encode_image(batch.to(device)).cpu().numpy())
    features = np.concatenate(chunks).astype(np.float16)

``` 

To run this code and get the embeddings locally:

`python extract_clip_features.py`

### Training Process

#### Preprocessing 

We start our training process by loading in our embedding matrix `clip_vitb16.npy` and the labels `index.csv`. 



**Parameter Choices**
- For initial hparam choices, we used Claude Opus recomnedations. So far we have done minimal hparam tuning, but plan to set up a grid search due to how fast the model trains. 

**Loss Functions**


**Metrics used for Eval**: TODO

#### Initial Results

Present initial results, discussing alignment with expectations and project objectives.

## 4. Results Interpretation and Analysis:
Analyze the baseline model’s performance, using appropriate metrics and visualizations.

Assess strengths and weaknesses of the model, proposing improvements for the next iteration.

Propose future directions: what methods, models, and adjustments do you expect to incorporate for the final project.  These should be grounded based on insights gathered from the data, baseline models, and/or outside resources.

### Euclidean vs. Hyperbolic Validation Results

We evaluated both models on the **24,421-image validation split** using the metrics proposed in the evaluation section of the pipeline document: tree distortion, dendrogram agreement, kNN retrieval consistency, style-classification accuracy, and Fr\u00e9chet-mean interpolation stability. Both models use frozen CLIP ViT-B/16 image features and an 8-dimensional learned embedding head; the Euclidean model serves as the baseline, while the hyperbolic model replaces the Euclidean tail with a Poincar\u00e9-ball embedding and Riemannian prototype optimization.

| Metric | Euclidean d=8 | Hyperbolic d=8 | Better |
|---|---:|---:|---|
| Top-1 accuracy | 54.3% | 47.3% | Euclidean |
| Top-5 accuracy | 92.4% | 89.8% | Euclidean |
| Balanced accuracy | 37.2% | 30.0% | Euclidean |
| Class-center/tree Spearman | 0.292 | 0.189 | Euclidean |
| Average tree distortion | 1.782 | 1.856 | Euclidean |
| Worst-case tree distortion | 5.339 | 6.599 | Euclidean |
| Dendrogram cluster F1 | 0.051 | 0.000 | Euclidean |
| Sibling recall@5 | 0.199 | 0.219 | Hyperbolic |
| Sibling recall@10 | 0.281 | 0.328 | Hyperbolic |
| Cousin recall@5 | 0.287 | 0.315 | Hyperbolic |
| Cousin recall@10 | 0.365 | 0.405 | Hyperbolic |
| Cubism Fr\u00e9chet-mean nearest-prototype accuracy | 0.91 | 0.39 | Euclidean |

The broadest conclusion from this first matched experiment is that the **Euclidean baseline is currently stronger on most global metrics**, while the **hyperbolic model shows a narrower local-neighborhood advantage**. In particular, Euclidean outperforms hyperbolic on top-1 and top-5 classification accuracy, balanced accuracy, correlation between centroid distances and the hand-built style tree, tree distortion, dendrogram agreement, and interpolation stability. This means that at the current embedding dimension (`d=8`) and training budget (10 epochs), the hyperbolic model is **not yet** recovering more of the global art-historical hierarchy than the Euclidean baseline.

That said, the hyperbolic model is not uniformly worse. Its strongest relative result is in **kNN retrieval consistency**: hyperbolic embeddings achieve better sibling and cousin recall at both `k=5` and `k=10`. This suggests that although the hyperbolic model is not yet organizing the full style space as well as the Euclidean baseline, it may already be producing **stronger local subtree neighborhoods**. That is an important clue for the final project, because one reasonable outcome is that hyperbolic geometry helps local branching structure before it improves global classification or full-tree reconstruction.

The Euclidean baseline remains the stronger overall model at this stage. Its validation top-1 accuracy reaches **54.3%**, compared with **47.3%** for the hyperbolic model, and its balanced accuracy is also substantially higher (**37.2%** vs. **30.0%**). This matters because the balanced metric reduces the influence of the largest classes, so the gap is not just a byproduct of Euclidean doing better on Impressionism-like majority classes. The centroid/tree Spearman correlation of **0.292** for Euclidean versus **0.189** for hyperbolic likewise suggests that the Euclidean embedding is preserving the hand-crafted hierarchy more faithfully at the global level, at least under the current training objective.

Tree distortion and dendrogram agreement reinforce that picture. Euclidean achieves a lower average multiplicative distortion (**1.782** vs. **1.856**) and a lower worst-case distortion (**5.339** vs. **6.599**), and it recovers at least a small amount of the ground-truth hierarchy in agglomerative clustering (`F1 = 0.051`), whereas the current hyperbolic run recovers none of the exact internal hierarchy clusters (`F1 = 0.000`). Looking into the worst-case distorted pairs is informative: the Euclidean model struggles most on distant cross-branch relationships such as New Realism versus Post-Impressionism, while the hyperbolic model’s worst distortions are more surprising because some **adjacent styles** (for example Abstract Expressionism versus Color Field Painting) are pushed far apart. That indicates the current hyperbolic objective is over-separating some fine-grained relatives that the hierarchy says should remain close.

The Fr\u00e9chet-mean interpolation experiment tells a similar story. For Euclidean, a mean over 8 randomly sampled Cubism embeddings is nearest to the Cubism prototype in **91%** of trials, with a positive margin over the closest competing prototype and a relatively small average distance to held-out Cubism samples. For hyperbolic, the same procedure succeeds in only **39%** of trials, and the average margin becomes slightly negative, meaning the mean often lands closer to another style’s prototype than to Cubism itself. So for now, Euclidean averaging appears much more stable than hyperbolic Fr\u00e9chet averaging in this setup.

Overall, the current evidence does **not** support the claim that hyperbolic geometry is already outperforming Euclidean geometry on the main project question. However, the negative result is still meaningful. It suggests that frozen CLIP features may already encode a great deal of style structure in a form that a small Euclidean head can exploit effectively, and that simply swapping in hyperbolic geometry is not enough by itself. It also suggests that if hyperbolic embeddings are going to win, the gain may emerge under different conditions: lower-dimensional bottlenecks, more training, better curvature tuning, stronger hierarchy-aware losses, or metrics that emphasize local branching more than top-1 classification.

The most natural next steps for the final project are therefore: (1) sweep embedding dimension and curvature rather than testing only `d=8, c=1`; (2) train the hyperbolic model longer and tune its optimization separately rather than reusing the Euclidean budget; (3) replace or augment the current prototype-classification objective with a more explicitly hierarchy-aware loss; and (4) continue comparing both geometries under the same evaluation harness rather than relying only on classification accuracy. Even though Euclidean currently wins overall, the hyperbolic advantage on subtree retrieval means it is still worth exploring further.

#### Validation Confusion Matrices

Euclidean baseline:

![Euclidean Confusion Matrix](../data/runs/euclidean_d8_e10/eval_val_section2/confusion_matrix.png)

Hyperbolic baseline:

![Hyperbolic Confusion Matrix](../data/runs/hyperbolic_d8_e10/eval_val_section2/confusion_matrix.png)

## 5. Final Model Pipeline Setup:
Outline steps for a pipeline for your final models, describing each component from data preprocessing to evaluation.

The pipeline can be laid out as a diagram, as an algorithmic outline, or with code or pseudocode.,,s

Document assumptions, parameter choices, and preliminary tuning considerations.